<a href="https://colab.research.google.com/github/Yatharth7651/Sales_Forecasting/blob/main/Sales_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install streamlit yfinance scikit-learn plotly statsmodels pyngrok xgboost ta --quiet

In [8]:
!streamlit run app.py &> /content/logs.txt &

In [9]:
from pyngrok import ngrok
import time

time.sleep(5)  # wait for streamlit

ngrok.kill()   # 🔥 avoid error

public_url = ngrok.connect(8501)

print("=" * 50)
print("🚀 App is LIVE at:", public_url)
print("=" * 50)

🚀 App is LIVE at: NgrokTunnel: "https://tuberculose-retha-unerased.ngrok-free.dev" -> "http://localhost:8501"


In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime
from model import run_forecasting_pipeline
from data_fetcher import get_popular_tickers

st.set_page_config(
    page_title="Sales Forecasting System",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

def render_sidebar():
    st.sidebar.title("📈 Sales Forecasting")
    st.sidebar.markdown("---")

    popular = get_popular_tickers()
    ticker_options = ["Type manually..."] + list(popular.keys())

    selected = st.sidebar.selectbox(
        "Select a company:",
        ticker_options,
        format_func=lambda x: f"{x} - {popular[x]}" if x in popular else x
    )

    if selected == "Type manually...":
        ticker = st.sidebar.text_input(
            "Enter Ticker Symbol:",
            placeholder="e.g., AAPL, GOOGL, TCS.NS"
        ).strip().upper()
    else:
        ticker = selected
        st.sidebar.info(f"Selected: **{popular[ticker]}**")

    st.sidebar.markdown("---")

    period = st.sidebar.select_slider(
        "Historical Data Period:",
        options=['6mo', '1y', '2y', '5y', '10y', 'max'],
        value='2y'
    )

    forecast_days = st.sidebar.slider(
        "Forecast Days:",
        min_value=7,
        max_value=90,
        value=30,
        step=7
    )

    run_forecast = st.sidebar.button(
        "🚀 Generate Forecast",
        use_container_width=True,
        type="primary"
    )

    return ticker, period, forecast_days, run_forecast


def render_results(result):
    ticker = result['ticker']
    info = result['company_info']
    stats = result['summary_stats']
    model_results = result['model_results']
    best_model = result['best_model']
    forecast_df = result['forecast_df']
    forecaster = result['forecaster']
    processed_data = result['processed_data']
    raw_data = result['raw_data']
    dates_test = result['dates_test']
    y_test = result['y_test']

    st.markdown(f"""
    <h1 style='text-align:center;'>📊 {info['name']}</h1>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)
    col1.metric("Current Price", f"Rs{stats['current_price']:,.2f}")
    col2.metric("Forecast Average", f"Rs{stats['forecast_avg']:,.2f}")
    col3.metric(
        "Expected Change",
        f"{stats['price_change_pct']}%",
        delta=f"{stats['price_change_pct']}%"
    )

    st.markdown("---")

    st.subheader("📈 Forecast")
    forecast_fig = forecaster.plot_forecast(processed_data, ticker)
    st.plotly_chart(forecast_fig, use_container_width=True)

    st.markdown("---")

    st.subheader("🏆 Model Performance")
    results_df = pd.DataFrame(model_results).T
    st.dataframe(results_df)

    st.markdown("---")

    st.subheader("📋 Forecast Data")
    display_df = forecast_df.copy()
    display_df['Date'] = display_df['Date'].dt.strftime('%Y-%m-%d')
    st.dataframe(display_df, use_container_width=True)

    csv = forecast_df.to_csv(index=False)
    st.download_button(
        label="Download Forecast CSV",
        data=csv,
        file_name=f"{ticker}_forecast_{datetime.now().strftime('%Y%m%d')}.csv",
        mime="text/csv"
    )

    st.markdown("---")

    st.subheader("📉 Raw Data")
    raw_display = raw_data.copy()
    raw_display['Date'] = raw_display['Date'].dt.strftime('%Y-%m-%d')
    st.dataframe(raw_display.tail(100), use_container_width=True)


def main():
    ticker, period, forecast_days, run_forecast = render_sidebar()

    if run_forecast:
        if not ticker:
            st.error("Please enter a ticker symbol!")
            return

        with st.spinner("Processing..."):
            try:
                result = run_forecasting_pipeline(
                    ticker=ticker,
                    period=period,
                    forecast_days=forecast_days
                )

                if 'error' in result:
                    st.error(result['error'])
                else:
                    st.session_state['result'] = result
                    render_results(result)

            except Exception as e:
                st.error(f"Error: {str(e)}")

    elif 'result' in st.session_state:
        render_results(st.session_state['result'])


if __name__ == "__main__":
    main()

Overwriting app.py


In [11]:
%%writefile data_fetcher.py
import yfinance as yf
import pandas as pd


def fetch_stock_data(ticker_symbol, period="2y"):
    try:
        ticker = yf.Ticker(ticker_symbol)
        data = ticker.history(period=period)

        if data.empty:
            return None, None, "No data found. Check ticker symbol."

        data.reset_index(inplace=True)

        try:
            info = ticker.info
            company_info = {
                'name': info.get('longName', ticker_symbol),
                'sector': info.get('sector', 'N/A'),
                'industry': info.get('industry', 'N/A'),
                'currency': info.get('currency', 'USD'),
                'market_cap': info.get('marketCap', 'N/A'),
                'website': info.get('website', 'N/A'),
                'summary': info.get('longBusinessSummary', 'N/A')
            }
        except Exception:
            company_info = {
                'name': ticker_symbol,
                'sector': 'N/A',
                'industry': 'N/A',
                'currency': 'USD',
                'market_cap': 'N/A',
                'website': 'N/A',
                'summary': 'N/A'
            }

        return data, company_info, None

    except Exception as e:
        return None, None, str(e)


def get_popular_tickers():
    return {
        'AAPL': 'Apple Inc.',
        'GOOGL': 'Alphabet (Google)',
        'MSFT': 'Microsoft Corp.',
        'AMZN': 'Amazon.com Inc.',
        'TSLA': 'Tesla Inc.',
        'META': 'Meta (Facebook)',
        'NVDA': 'NVIDIA Corp.',
        'JPM': 'JPMorgan Chase',
        'WMT': 'Walmart Inc.',
        'NFLX': 'Netflix Inc.',
        'TCS.NS': 'TCS (India)',
        'RELIANCE.NS': 'Reliance (India)',
        'INFY.NS': 'Infosys (India)',
        'HDFCBANK.NS': 'HDFC Bank (India)',
        'WIPRO.NS': 'Wipro (India)',
        'ITC.NS': 'ITC Ltd (India)',
        'SBIN.NS': 'SBI (India)',
        'TATAMOTORS.NS': 'Tata Motors (India)'
    }

Overwriting data_fetcher.py


In [12]:
%%writefile model.py
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")


class SaleForecaster:

    def __init__(self):
        self.models = {}
        self.results = {}
        self.predictions = {}
        self.forecast_data = None
        self.feature_cols = []
        self.scaler = StandardScaler()
        self.date_col = 'Date'
        self.target_col = 'Close'

    # ─────────────────────────────────────────────
    # TECHNICAL INDICATORS
    # ─────────────────────────────────────────────
    def add_technical_indicators(self, df):

        # ── EMA ──────────────────────────────────
        df['EMA_9']  = df['Close'].ewm(span=9,  adjust=False).mean()
        df['EMA_21'] = df['Close'].ewm(span=21, adjust=False).mean()

        # ── RSI ──────────────────────────────────
        delta = df['Close'].diff()
        gain  = delta.clip(lower=0)
        loss  = -delta.clip(upper=0)
        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()
        rs = avg_gain / (avg_loss + 1e-10)
        df['RSI'] = 100 - (100 / (1 + rs))

        # ── MACD ─────────────────────────────────
        ema12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema26 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD']        = ema12 - ema26
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

        # ── Bollinger Bands ───────────────────────
        sma20 = df['Close'].rolling(20).mean()
        std20 = df['Close'].rolling(20).std()
        df['BB_Upper'] = sma20 + (2 * std20)
        df['BB_Lower'] = sma20 - (2 * std20)
        df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']

        # ── Volume features ───────────────────────
        if 'Volume' in df.columns:
            df['Volume_MA7']  = df['Volume'].rolling(7).mean()
            df['Volume_MA21'] = df['Volume'].rolling(21).mean()

        # ── Price momentum ────────────────────────
        df['Price_Change_1d'] = df['Close'].pct_change(1)
        df['Price_Change_5d'] = df['Close'].pct_change(5)
        df['Price_Change_10d']= df['Close'].pct_change(10)

        # ── High/Low range ────────────────────────
        if 'High' in df.columns and 'Low' in df.columns:
            df['HL_Range']     = df['High'] - df['Low']
            df['HL_Range_MA7'] = df['HL_Range'].rolling(7).mean()

        return df

    # ─────────────────────────────────────────────
    # PREPARE DATA
    # ─────────────────────────────────────────────
    def prepare_data(self, df):

        df = df.copy()
        df['Date'] = pd.to_datetime(df['Date'])

        if df['Date'].dt.tz is not None:
            df['Date'] = df['Date'].dt.tz_localize(None)

        df.sort_values('Date', inplace=True)

        # ── Date features ─────────────────────────
        df['Year']      = df['Date'].dt.year
        df['Month']     = df['Date'].dt.month
        df['Day']       = df['Date'].dt.day
        df['DayOfWeek'] = df['Date'].dt.dayofweek
        df['Quarter']   = df['Date'].dt.quarter
        df['WeekOfYear']= df['Date'].dt.isocalendar().week.astype(int)

        # ── Lag features ──────────────────────────
        for lag in [1, 3, 7, 14, 21, 30]:
            df[f'Lag_{lag}'] = df['Close'].shift(lag)

        # ── Rolling stats ─────────────────────────
        for window in [7, 14, 21, 30]:
            df[f'Rolling_Mean_{window}'] = df['Close'].rolling(window).mean()
            df[f'Rolling_Std_{window}']  = df['Close'].rolling(window).std()

        # ── Technical indicators ──────────────────
        df = self.add_technical_indicators(df)

        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)

        return df

    # ─────────────────────────────────────────────
    # SPLIT DATA
    # ─────────────────────────────────────────────
    def split_data(self, df, test_ratio=0.2):

        split_index = int(len(df) * (1 - test_ratio))

        # All feature columns (exclude Date and Close)
        exclude = ['Date', 'Close', 'Open', 'High', 'Low',
                   'Volume', 'Dividends', 'Stock Splits']
        self.feature_cols = [
            c for c in df.columns if c not in exclude
        ]

        X = df[self.feature_cols]
        y = df['Close']

        X_train = X[:split_index]
        X_test  = X[split_index:]
        y_train = y[:split_index]
        y_test  = y[split_index:]
        dates_test = df['Date'][split_index:]

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled  = self.scaler.transform(X_test)

        return (X_train, X_test, y_train, y_test,
                dates_test, X_train_scaled, X_test_scaled)

    # ─────────────────────────────────────────────
    # TRAIN MODELS
    # ─────────────────────────────────────────────
    def train_models(self, X_train, y_train,
                     X_train_scaled, y_train_s=None):

        # Linear Regression uses scaled features
        lr = LinearRegression()
        lr.fit(X_train_scaled, y_train)
        self.models['Linear Regression'] = ('scaled', lr)

        # Tree-based models use raw features
        rf = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        self.models['Random Forest'] = ('raw', rf)

        gb = GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=42
        )
        gb.fit(X_train, y_train)
        self.models['Gradient Boosting'] = ('raw', gb)

        # XGBoost
        xgb = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        )
        xgb.fit(X_train, y_train)
        self.models['XGBoost'] = ('raw', xgb)

    # ─────────────────────────────────────────────
    # EVALUATE
    # ─────────────────────────────────────────────
    def evaluate(self, X_test, y_test, X_test_scaled):

        for name, (ftype, model) in self.models.items():

            X_input = X_test_scaled if ftype == 'scaled' else X_test
            preds   = model.predict(X_input)

            mape = np.mean(
                np.abs((np.array(y_test) - preds) /
                       (np.array(y_test) + 1e-10))
            ) * 100

            self.results[name] = {
                "MAE":      round(mean_absolute_error(y_test, preds), 4),
                "RMSE":     round(np.sqrt(mean_squared_error(y_test, preds)), 4),
                "R2_Score": round(r2_score(y_test, preds), 4),
                "MAPE(%)":  round(mape, 4)
            }
            self.predictions[name] = preds

    # ─────────────────────────────────────────────
    # BEST MODEL
    # ─────────────────────────────────────────────
    def get_best_model(self):
        best_name = max(
            self.results,
            key=lambda x: self.results[x]["R2_Score"]
        )
        ftype, model = self.models[best_name]
        return best_name, model, ftype

    # ─────────────────────────────────────────────
    # FORECAST FUTURE
    # ─────────────────────────────────────────────
    def forecast_future(self, df, days=30):

        best_name, best_model, ftype = self.get_best_model()

        last_date  = df['Date'].max()
        future_dates = pd.date_range(
            start=last_date + pd.Timedelta(days=1),
            periods=days,
            freq='B'
        )

        future_predictions = []
        last_close = df['Close'].iloc[-1]

        # Seed rolling windows from last known values
        recent_closes = list(df['Close'].tail(30))

        for date in future_dates:

            # Rolling stats from recent window
            rc = np.array(recent_closes)
            row = {
                'Year':      date.year,
                'Month':     date.month,
                'Day':       date.day,
                'DayOfWeek': date.dayofweek,
                'Quarter':   (date.month - 1) // 3 + 1,
                'WeekOfYear': date.isocalendar()[1],
                'Lag_1':  rc[-1]  if len(rc) >= 1  else last_close,
                'Lag_3':  rc[-3]  if len(rc) >= 3  else last_close,
                'Lag_7':  rc[-7]  if len(rc) >= 7  else last_close,
                'Lag_14': rc[-14] if len(rc) >= 14 else last_close,
                'Lag_21': rc[-21] if len(rc) >= 21 else last_close,
                'Lag_30': rc[-30] if len(rc) >= 30 else last_close,
                'Rolling_Mean_7':  rc[-7:].mean()  if len(rc) >= 7  else rc.mean(),
                'Rolling_Std_7':   rc[-7:].std()   if len(rc) >= 7  else rc.std(),
                'Rolling_Mean_14': rc[-14:].mean() if len(rc) >= 14 else rc.mean(),
                'Rolling_Std_14':  rc[-14:].std()  if len(rc) >= 14 else rc.std(),
                'Rolling_Mean_21': rc[-21:].mean() if len(rc) >= 21 else rc.mean(),
                'Rolling_Std_21':  rc[-21:].std()  if len(rc) >= 21 else rc.std(),
                'Rolling_Mean_30': rc[-30:].mean() if len(rc) >= 30 else rc.mean(),
                'Rolling_Std_30':  rc[-30:].std()  if len(rc) >= 30 else rc.std(),
                'EMA_9':  pd.Series(rc).ewm(span=9,  adjust=False).mean().iloc[-1],
                'EMA_21': pd.Series(rc).ewm(span=21, adjust=False).mean().iloc[-1],
                'RSI':    50.0,   # neutral placeholder
                'MACD':   0.0,
                'MACD_Signal': 0.0,
                'BB_Upper': rc.mean() + 2 * rc.std(),
                'BB_Lower': rc.mean() - 2 * rc.std(),
                'BB_Width': 4 * rc.std(),
                'Price_Change_1d':  (rc[-1] - rc[-2]) / (rc[-2] + 1e-10) if len(rc) >= 2  else 0,
                'Price_Change_5d':  (rc[-1] - rc[-6]) / (rc[-6] + 1e-10) if len(rc) >= 6  else 0,
                'Price_Change_10d': (rc[-1] - rc[-11])/ (rc[-11]+ 1e-10) if len(rc) >= 11 else 0,
            }

            # Volume features (use last known if available)
            if 'Volume_MA7' in self.feature_cols:
                row['Volume_MA7']  = df['Volume'].tail(7).mean()
                row['Volume_MA21'] = df['Volume'].tail(21).mean()

            if 'HL_Range' in self.feature_cols:
                row['HL_Range']     = df['HL_Range'].iloc[-1] if 'HL_Range' in df.columns else 0
                row['HL_Range_MA7'] = df['HL_Range'].tail(7).mean() if 'HL_Range' in df.columns else 0

            X_new = pd.DataFrame([row])[self.feature_cols]

            if ftype == 'scaled':
                X_new = self.scaler.transform(X_new)

            pred = best_model.predict(X_new)[0]

            future_predictions.append({
                "Date": date,
                "Predicted_Sale": round(pred, 2)
            })

            recent_closes.append(pred)
            last_close = pred

        self.forecast_data = pd.DataFrame(future_predictions)
        return self.forecast_data, best_name

    # ─────────────────────────────────────────────
    # PLOT FORECAST
    # ─────────────────────────────────────────────
    def plot_forecast(self, df, ticker):

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=list(df['Date'].tail(60)),
            y=list(df['Close'].tail(60)),
            mode='lines',
            name='Historical',
            line=dict(color='#00b4d8', width=2)
        ))

        if self.forecast_data is not None and not self.forecast_data.empty:
            fig.add_trace(go.Scatter(
                x=list(self.forecast_data['Date']),
                y=list(self.forecast_data['Predicted_Sale']),
                mode='lines+markers',
                name='Forecast',
                line=dict(color='#f77f00', width=2),
                marker=dict(size=5)
            ))
            fig.add_vline(
                x=str(df['Date'].max()),
                line_dash="dash",
                line_color="gray"
            )

        fig.update_layout(
            title=f"{ticker} - Sales Forecast",
            template="plotly_dark",
            height=500,
            xaxis_title="Date",
            yaxis_title="Price"
        )

        return fig


# ─────────────────────────────────────────────
# COMPLETE PIPELINE
# ─────────────────────────────────────────────
def run_forecasting_pipeline(ticker, period='2y', forecast_days=30):

    from data_fetcher import fetch_stock_data

    data, company_info, error = fetch_stock_data(ticker, period)

    if error:
        return {"error": error}
    if len(data) < 60:
        return {"error": "Not enough data. Try a longer period."}

    forecaster = SaleForecaster()
    raw_data   = data.copy()
    df         = forecaster.prepare_data(data)

    (X_train, X_test, y_train, y_test,
     dates_test, X_train_scaled, X_test_scaled) = forecaster.split_data(df)

    forecaster.train_models(X_train, y_train, X_train_scaled)
    forecaster.evaluate(X_test, y_test, X_test_scaled)

    forecast_df, best_model = forecaster.forecast_future(df, forecast_days)

    current_price = round(data['Close'].iloc[-1], 2)
    forecast_avg  = round(forecast_df['Predicted_Sale'].mean(), 2)

    summary_stats = {
        "current_price":    current_price,
        "avg_price":        round(data['Close'].mean(), 2),
        "forecast_avg":     forecast_avg,
        "price_change_pct": round(
            ((forecast_avg - current_price) / current_price) * 100, 2
        )
    }

    return {
        "ticker":       ticker,
        "forecaster":   forecaster,
        "processed_data": df,
        "forecast_df":  forecast_df,
        "summary_stats": summary_stats,
        "best_model":   best_model,
        "model_results": forecaster.results,
        "company_info": company_info,
        "raw_data":     raw_data,
        "dates_test":   dates_test,
        "y_test":       y_test
    }

Overwriting model.py


In [13]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
time.sleep(3)
print("✅ All cleared!")

✅ All cleared!


In [14]:
from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token("3BSy4LuAHZfG8zliJIquGWQ1a51_2JJ1Y9qAg6NRCrEopoZ9A")

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

public_url = ngrok.connect(8501)
print("=" * 50)
print(f"🚀 App is LIVE at: {public_url}")
print("=" * 50)

🚀 App is LIVE at: NgrokTunnel: "https://tuberculose-retha-unerased.ngrok-free.dev" -> "http://localhost:8501"
